# Python Data Visualization Assignment 2

**Dataset:** Seaborn `taxis` dataset  
**Libraries:** Pandas, Matplotlib, Seaborn

This notebook follows the supplied assignment brief. It covers loading the taxis data, handling missing values, five Matplotlib/Pandas visualizations, and five Seaborn visualizations.

## 1. Load the Taxis Dataset

Load the built-in `taxis` dataset from Seaborn and inspect its structure.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df = sns.load_dataset("taxis")

print("Dataset shape:", df.shape)
display(df.head())
df.info()

## 2. Explore and Handle Missing Values

Identify columns with missing data. Numeric columns are imputed with the median and categorical columns with the mode. Rows without a pickup timestamp are removed because that timestamp is essential for the time-series chart.

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
})
missing_summary = missing_summary[missing_summary["Missing Count"] > 0].sort_values("Missing Count", ascending=False)
display(missing_summary)

In [ ]:
if not missing_summary.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.heatmap(df.isna(), cbar=False, yticklabels=False, ax=ax)
    ax.set_title("Missing Values in Taxi Dataset (Before Imputation)")
    ax.set_xlabel("Dataset columns")
    plt.tight_layout()
    ax.figure.savefig(OUTPUT_DIR / "00_missing_values_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# Convert pickup to datetime and remove rows where the required timestamp is missing.
rows_before_cleaning = len(df)
df["pickup"] = pd.to_datetime(df["pickup"], errors="coerce")
df = df.dropna(subset=["pickup"]).copy()

# Impute numeric columns with their median.
numeric_cols = df.select_dtypes(include="number").columns
for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

# Impute categorical/text columns with their mode.
categorical_cols = df.select_dtypes(include=["object", "category", "string"]).columns
for col in categorical_cols:
    if df[col].isna().any():
        modes = df[col].mode(dropna=True)
        if not modes.empty:
            df[col] = df[col].fillna(modes.iloc[0])

print("Missing values after treatment:")
display(df.isna().sum())
print("Rows removed:", rows_before_cleaning - len(df))
print("Cleaned dataset shape:", df.shape)

## 3. Visualizations using Matplotlib / Pandas Plot

### 3.1 Line Chart — Fare Over Time

Use pickup timestamp on the x-axis and fare on the y-axis. Aggregate fares by pickup time interval to make the trend readable.

In [ ]:
fare_over_time = (df.set_index("pickup")["fare"]
                  .resample("D").mean().dropna())

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(fare_over_time.index, fare_over_time.values, linewidth=1.8)
ax.set_title("Average Taxi Fare Over Time")
ax.set_xlabel("Pickup Timestamp")
ax.set_ylabel("Average Fare")
fig.autofmt_xdate()
plt.tight_layout()
ax.figure.savefig(OUTPUT_DIR / "01_fare_over_time_line_chart.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.2 Bar Chart — Total Fare by Pickup Borough

Group the data by `pickup_borough` and calculate the sum of `fare` for each borough.

In [ ]:
fare_by_borough = df.groupby("pickup_borough", observed=True)["fare"].sum().sort_values(ascending=False)

ax = fare_by_borough.plot(kind="bar", figsize=(9, 5), title="Total Fare by Pickup Borough")
ax.set_xlabel("Pickup Borough")
ax.set_ylabel("Total Fare")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
ax.figure.savefig(OUTPUT_DIR / "02_total_fare_by_borough_bar_chart.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.3 Pie Chart — Trips by Payment Method

Each slice represents the number of trips for a payment method.

In [ ]:
payment_counts = df["payment"].value_counts()
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(payment_counts, labels=payment_counts.index.astype(str), autopct="%1.1f%%", startangle=90)
ax.set_title("Trip Distribution by Payment Method")
ax.axis("equal")
plt.tight_layout()
ax.figure.savefig(OUTPUT_DIR / "03_payment_method_pie_chart.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.4 Histogram — Distance Distribution

Customize the number of bins to show the distribution of trip distances.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df["distance"].dropna(), bins=30, edgecolor="black")
ax.set_title("Distribution of Taxi Trip Distance")
ax.set_xlabel("Distance")
ax.set_ylabel("Number of Trips")
plt.tight_layout()
ax.figure.savefig(OUTPUT_DIR / "04_trip_distance_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.5 Box Plot — Tip Amount by Pickup Borough

Use pickup borough as the categorical axis and tip as the numeric axis.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="pickup_borough", y="tip", ax=ax)
ax.set_title("Tip Amount Distribution by Pickup Borough")
ax.set_xlabel("Pickup Borough")
ax.set_ylabel("Tip Amount")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
ax.figure.savefig(OUTPUT_DIR / "05_tip_by_borough_box_plot.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Visualizations using Seaborn

### 4.1 Count Plot — Number of Trips by Pickup Borough

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.countplot(data=df, x="pickup_borough", order=df["pickup_borough"].value_counts().index, ax=ax)
ax.set_title("Number of Taxi Trips by Pickup Borough")
ax.set_xlabel("Pickup Borough")
ax.set_ylabel("Number of Trips")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
ax.figure.savefig(OUTPUT_DIR / "06_trips_by_borough_count_plot.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.2 Scatter Plot — Distance vs Fare

Color points by `pickup_borough` to compare trips from different boroughs.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x="distance", y="fare", hue="pickup_borough", alpha=0.65, ax=ax)
ax.set_title("Relationship Between Trip Distance and Fare")
ax.set_xlabel("Distance")
ax.set_ylabel("Fare")
plt.tight_layout()
ax.figure.savefig(OUTPUT_DIR / "07_distance_vs_fare_scatter_plot.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.3 Heatmap — Correlation Between Numeric Variables

Use the correlation matrix for distance, fare, tip, tolls, and total.

In [ ]:
corr_cols = ["distance", "fare", "tip", "tolls", "total"]
correlation_matrix = df[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Heatmap of Taxi Variables")
plt.tight_layout()
ax.figure.savefig(OUTPUT_DIR / "08_numeric_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.4 Pair Plot — Pairwise Relationships

Create pairwise plots for distance, fare, tip, and total. Use `pickup_zone` as the hue. To keep the plot readable, this exercise uses the five most frequent pickup zones; the selection is a plotting convenience and does not alter the cleaned dataset.

In [ ]:
pair_cols = ["distance", "fare", "tip", "total", "pickup_zone"]
top_zones = df["pickup_zone"].value_counts().head(5).index
pair_df = df.loc[df["pickup_zone"].isin(top_zones), pair_cols].dropna()

pair_plot = sns.pairplot(pair_df, vars=["distance", "fare", "tip", "total"], hue="pickup_zone", diag_kind="hist", corner=True)
pair_plot.fig.suptitle("Pairwise Relationships Across Top 5 Pickup Zones", y=1.02)
pair_plot.savefig(OUTPUT_DIR / "09_pickup_zone_pair_plot.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.5 Violin Plot — Fare by Payment Method

Use payment method as the categorical axis and fare as the numeric axis.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.violinplot(data=df, x="payment", y="fare", inner="quartile", ax=ax)
ax.set_title("Fare Distribution by Payment Method")
ax.set_xlabel("Payment Method")
ax.set_ylabel("Fare")
plt.tight_layout()
ax.figure.savefig(OUTPUT_DIR / "10_fare_by_payment_method_violin_plot.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Findings and Interpretation

Complete the following after reviewing your charts:

1. Which pickup borough has the highest total fare?
2. Which payment method accounts for the largest number of trips?
3. What pattern do you observe between trip distance and fare?
4. Which numeric variables show the strongest correlation?
5. How does the fare distribution vary across payment methods?
6. What differences are visible between pickup zones in the pair plot?

**Important:** Correlation and visual association do not prove causation. Mention any limitations you notice in the dataset or charts.

In [ ]:
# Add your written observations below as Python comments or in a new Markdown cell.
# Example:
# 1. Highest total fare borough:
# 2. Most common payment method:
# 3. Distance-fare pattern:
# 4. Strongest correlation:
# 5. Fare differences by payment method:
# 6. Pickup-zone observations:

## 6. Conclusion

Summarize what the visualizations reveal about taxi trips, fares, distances, tips, payment methods, and pickup locations. Include at least one limitation of the analysis.